# Filter positions by tissue quality

Drops FOVs with damaged tissue (nuclei whose DAPI has leaked into the
surrounding tissue, which likely gives no good signal) before the full
acquisition. Used by `tumor_disk`.

Run order: 02 (positions) -> `02c_create_focus_test_dave_config.ipynb`, with
pipeline.yaml's `focus_test_frames` > 0 so every FOV gets a 405 + blank
movie -> run that recipe on the microscope -> **this notebook** -> 03 and 04
on the kept FOVs.

1. Each FOV's focus-test DAPI frame and dark level (blank frame), from the
   focus-test movies in `data/focus_test/`.
2. The acquisition's haze, from its empty FOVs.
3. The damaged % of each FOV with the image-only tissue-quality model
   (`MERci.analysis.tissue_quality`, from the BC561 analysis).
4. Keep a FOV unless its damaged % > `MAX_DAMAGED_PCT`; FOVs with no tissue
   are dropped when `DROP_NO_TISSUE`.

Outputs, in `positions/`:

- `positions_{POSITIONS_TAG}_all.txt`: the original positions, unchanged
  (the focus-test FOV list; FOV i = focus-test movie i).
- `positions_{POSITIONS_TAG}.txt`: the kept FOVs, which every later
  notebook reads. Rerunning this notebook starts again from `_all`.
- `tissue_quality_{POSITIONS_TAG}.csv`: per FOV (index of `_all`): x, y,
  damaged %, tissue windows, kept.
- The map figure, under `SAMPLE_DIR/figures/MERci/before_imaging/`.

Validation (BC561): window AUC 0.990 out of fold on BC553disk (0.985 at the
z 3 µm focus-test plane); on BC555's focus test, 92% of tissue FOVs > 50%
damaged, as the user judged by eye. The model was trained on ST2 60x DAPI
only; on another microscope the results are unvalidated.

In [ ]:
import os
import sys
import shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.collections import PatchCollection

MERCI_DIR  = Path(os.getcwd()).parent.parent.parent  # MERci/ (notebook lives in MERci/notebooks/before_imaging/regular/)
SAMPLE_DIR = MERCI_DIR.parent
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.experiment_info import resolve_sample_identity, positions_file_tag
from MERci.common.io import load_positions, save_positions_array, read_image_frames
from MERci.acquisition.configs import find_frame_table_for_hal_config, apply_microscope_orientation
from MERci.acquisition.merlin_config import load_microscope_parameters
from MERci.acquisition.pipeline_config import load_pipeline_config
from MERci.analysis.tissue_quality import empty_fov_level, haze_level, load_model, score_fov, UM_PX
from MERci.plots.experiment_plots import get_merci_figures_dir
from MERci.progress_display import ProgressReporter

# PIPELINE_ID names this variant's own pipeline.yaml -- see notebook 01's own
# comment on PIPELINE_ID/PIPELINE_CONFIG for the rationale.
PIPELINE_ID     = "tumor_disk"   # EDIT ME -- one of the ids in pipeline_export.PIPELINES
PIPELINE_CONFIG = load_pipeline_config(MERCI_DIR / "data" / "pipelines" / f"{PIPELINE_ID}_pipeline.yaml")

NOTEBOOK_NAME = "02d_filter_positions_by_tissue_quality"
SETTINGS_DIR  = SAMPLE_DIR / "settings"
METADATA_DIR  = SAMPLE_DIR / "metadata"
POSITIONS_DIR = SAMPLE_DIR / "positions"
CACHE_DIR     = SAMPLE_DIR / "analysis" / "cache" / NOTEBOOK_NAME
FIGURES_DIR   = get_merci_figures_dir(SAMPLE_DIR, "before_imaging", NOTEBOOK_NAME)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(MERCI_DIR)
POSITIONS_TAG = positions_file_tag(SAMPLE_NAME, IMAGING_DIR)
MICROSCOPE    = PIPELINE_CONFIG.microscope
print(f"SAMPLE_DIR    : {SAMPLE_DIR}")
print(f"POSITIONS_TAG : {POSITIONS_TAG}")
print(f"MICROSCOPE    : {MICROSCOPE} {PIPELINE_CONFIG.objective}")

## Parameters

In [ ]:
MAX_DAMAGED_PCT = 50      # drop a FOV whose damaged % is above this
DROP_NO_TISSUE  = True    # drop FOVs with no tissue (no 103 µm window with >= 20% tissue)
FOCUS_TEST_DIR  = SAMPLE_DIR / "data" / "focus_test"   # where 02c's recipe writes the movies

PLOT_TITLE_FONTSIZE  = 16
PLOT_LABEL_FONTSIZE  = 14
PLOT_TICK_FONTSIZE   = 12
PLOT_LEGEND_FONTSIZE = 12

## Inputs

The full FOV list is `positions_{POSITIONS_TAG}_all.txt` once this notebook
has run, else `positions_{POSITIONS_TAG}.txt`. Which focus-test frame is
DAPI (405) and which is blank comes from the focus-test frame table (the
one notebook 01 wrote for the focus-test HAL config). Movies are named as
Dave names them: FOV index zero-padded to the digits of the FOV count.
Orientation and binning come from the microscope's MERlin YAML, so the
images match the model's training frame (MERlin's orientation, ~0.357
µm per binned pixel).

In [ ]:
all_path   = POSITIONS_DIR / f"positions_{POSITIONS_TAG}_all.txt"
plain_path = POSITIONS_DIR / f"positions_{POSITIONS_TAG}.txt"
positions_path = all_path if all_path.exists() else plain_path
positions = load_positions(positions_path)
fovs = sorted(positions)
print(f"Positions: {positions_path.name} ({len(fovs)} FOVs)")
corrected_path = POSITIONS_DIR / f"positions_{POSITIONS_TAG}_corrected.txt"
if corrected_path.exists():
    print(f"WARNING: {corrected_path.name} exists and lists the FOVs before filtering -- "
          f"rerun notebooks/misc/correct_camera_rotation.ipynb after this notebook.")

# Focus-test frames: DAPI = the 405 row, dark = a blank (no color) row of its frame table
hal_configs = sorted(SETTINGS_DIR.glob("hal-config-*-focustest-*.xml"))
if not hal_configs:
    raise FileNotFoundError(f"No focus-test HAL config in {SETTINGS_DIR} -- run notebook 01 first.")
frame_table_path = find_frame_table_for_hal_config(hal_configs[-1], METADATA_DIR)
if frame_table_path is None:
    raise FileNotFoundError(f"No frame table in {METADATA_DIR} for {hal_configs[-1].name}.")
ft = pd.read_csv(frame_table_path, index_col=0)
if not (ft["color"] == 405).any() or not ft["color"].isna().any():
    raise ValueError(f"{frame_table_path.name} needs a 405 frame and a blank frame.")
DAPI_FRAME  = int(ft.index[ft["color"] == 405][0])
BLANK_FRAME = int(ft.index[ft["color"].isna()][0])
print(f"Frame table: {frame_table_path.name}  ->  DAPI frame {DAPI_FRAME}, blank frame {BLANK_FRAME}")

pad = len(str(len(fovs)))
movie = lambda f: FOCUS_TEST_DIR / f"hal-{MICROSCOPE.lower()}-focustest_{f:0{pad}d}{PIPELINE_CONFIG.file_type}"
missing = [f for f in fovs if not movie(f).exists()]
if missing:
    raise FileNotFoundError(f"{len(missing)} of {len(fovs)} focus-test movies missing in {FOCUS_TEST_DIR} "
                            f"(first: {movie(missing[0]).name}). Run 02c's recipe with focus_test_frames > 0.")

scope = load_microscope_parameters(MICROSCOPE, PIPELINE_CONFIG.objective)
ORIENTATION = {k: bool(scope[k]) for k in ("flip_horizontal", "flip_vertical", "transpose")}
FRAME_W, FRAME_H = scope["image_dimensions"]
BINNING = max(1, round(UM_PX / scope["microns_per_pixel"]))
FOV_UM  = FRAME_W * scope["microns_per_pixel"]
print(f"Binning {BINNING} -> {BINNING * scope['microns_per_pixel']:.4f} µm/px (model: {UM_PX} µm/px)")
if not (MICROSCOPE == "ST2" and PIPELINE_CONFIG.objective.upper() == "60X"):
    print(f"WARNING: the model was trained on ST2 60x DAPI only; results on {MICROSCOPE} "
          f"{PIPELINE_CONFIG.objective} are unvalidated.")


def read_fov(f):
    """(DAPI frame in MERlin's orientation, dark level) of one focus-test movie."""
    dapi, blank = read_image_frames(movie(f), [DAPI_FRAME, BLANK_FRAME], frame_width=FRAME_W, frame_height=FRAME_H)
    return apply_microscope_orientation(dapi, **ORIENTATION), float(np.median(blank))

## Haze

The acquisition's haze = median signal of its empty FOVs (tissue < 1%),
above the dark level. DAPI between nuclei is the main sign of damage, so an
unmeasured haze reads as damage (BC561: haze 20 counts too low moved the
FOV damaged % by a median 9 points). With fewer than 3 empty FOVs the
estimate is weak.

In [ ]:
# Calculation: each FOV's empty level (None = has tissue); cached per FOV list
levels_path = CACHE_DIR / "empty_levels.csv"
levels = pd.read_csv(levels_path, index_col="fov") if levels_path.exists() else None
if levels is None or list(levels.index) != fovs:
    rows = []
    reporter = ProgressReporter(total=len(fovs), label="Empty-FOV levels")
    for f in reporter.wrap(fovs):
        img, dark = read_fov(f)
        rows.append({"fov": f, "level": empty_fov_level(img, dark, BINNING)})
    levels = pd.DataFrame(rows).set_index("fov")
    levels.to_csv(levels_path)

HAZE = haze_level(levels["level"].dropna().tolist())   # NaN = FOV with tissue (None before the csv)
n_empty = int(levels["level"].notna().sum())
print(f"Empty FOVs: {n_empty}  ->  haze {HAZE:.1f} counts above dark")
if n_empty < 3:
    print(f"WARNING: only {n_empty} empty FOV(s) -- the haze estimate is weak, and an "
          f"underestimated haze reads as damage. Image a few FOVs off tissue if possible.")

## Damaged % per FOV

Each FOV's DAPI is cut into 103 µm windows (3 x 3 per ST2 FOV); windows with
at least 20% tissue get P(damaged). Damaged % = tissue-weighted share of
windows with P > 0.5. No window = no tissue.

In [ ]:
# Calculation: damaged % of every FOV (cached; recomputed if the FOVs or the haze change)
quality_path = CACHE_DIR / "fov_quality.csv"
quality = pd.read_csv(quality_path, index_col="fov") if quality_path.exists() else None
if quality is None or list(quality.index) != fovs or not np.isclose(quality["haze"].iloc[0], HAZE):
    model = load_model("image")
    rows = []
    reporter = ProgressReporter(total=len(fovs), label="Scoring FOVs")
    for f in reporter.wrap(fovs):
        img, dark = read_fov(f)
        r = score_fov(img, dark, HAZE, binning=BINNING, model=model)
        rows.append({"fov": f, "damaged_pct": r.damaged_pct, "tissue_windows": r.n_tissue_windows, "haze": HAZE})
    quality = pd.DataFrame(rows).set_index("fov")
    quality.to_csv(quality_path)
print(f"Scored {len(quality)} FOVs; {int((quality.tissue_windows > 0).sum())} with tissue")

## Keep or drop

In [ ]:
table = pd.DataFrame({"x": [positions[f][0] for f in fovs], "y": [positions[f][1] for f in fovs]}, index=fovs)
table.index.name = "fov"
table = table.join(quality[["damaged_pct", "tissue_windows"]])
no_tissue = table.tissue_windows == 0
damaged = table.damaged_pct > MAX_DAMAGED_PCT
table["kept"] = ~damaged & ~(no_tissue & DROP_NO_TISSUE)
if not table.kept.any():
    raise ValueError("Every FOV would be dropped -- check the haze and MAX_DAMAGED_PCT before writing positions.")

n_drop = int((~table.kept).sum())
print(f"FOVs: {len(table)} total, {int(table.kept.sum())} kept, {n_drop} dropped "
      f"({int(damaged.sum())} damaged > {MAX_DAMAGED_PCT}%, "
      f"{int(no_tissue.sum())} no tissue{'' if DROP_NO_TISSUE else ' (kept)'})")
print(f"Imaging time saved: {100 * n_drop / len(table):.1f}% of every round "
      f"(every round images every FOV; notebook 04 prints the hours from the kept FOVs)")

In [ ]:
# Write: the original list (once, unchanged) and the kept FOVs
if not all_path.exists():
    shutil.copy2(plain_path, all_path)
save_positions_array(table.loc[table.kept, ["x", "y"]].values, plain_path)
table.to_csv(POSITIONS_DIR / f"tissue_quality_{POSITIONS_TAG}.csv")
print(f"Original : {all_path}")
print(f"Kept     : {plain_path} ({int(table.kept.sum())} FOVs)")
print(f"Table    : {POSITIONS_DIR / f'tissue_quality_{POSITIONS_TAG}.csv'}")
print("Next: notebooks 03 and 04 (they read the kept FOVs). Notebook 02's fov_layout figure still shows every FOV.")

## Map

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
cmap = plt.get_cmap("magma_r")
squares = [Rectangle((r.x - FOV_UM / 2, r.y - FOV_UM / 2), FOV_UM, FOV_UM) for r in table.itertuples()]
pc = PatchCollection(squares, cmap=cmap, edgecolor="none")
pc.set_array(table.damaged_pct.values)
pc.set_clim(0, 100)
ax.add_collection(pc)
ax.add_collection(PatchCollection([s for s, nt in zip(squares, no_tissue) if nt],
                                  facecolor="#cccccc", edgecolor="none"))
ax.add_collection(PatchCollection([s for s, k in zip(squares, table.kept) if not k],
                                  facecolor="none", edgecolor="#00c8ff", linewidth=1.5))
ax.autoscale_view()
ax.set_aspect("equal")
cb = fig.colorbar(pc, ax=ax, shrink=0.7)
cb.set_label("damaged % of the FOV", fontsize=PLOT_LABEL_FONTSIZE)
cb.ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
ax.set_xlabel("stage x (µm)", fontsize=PLOT_LABEL_FONTSIZE)
ax.set_ylabel("stage y (µm)", fontsize=PLOT_LABEL_FONTSIZE)
ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
ax.set_title(f"{POSITIONS_TAG}\n{int(table.kept.sum())} kept, {n_drop} dropped (cyan outline) of {len(table)} FOVs, "
             f"{100 * n_drop / len(table):.0f}% imaging time saved\n"
             f"grey = no tissue; dropped above {MAX_DAMAGED_PCT}% damaged; haze {HAZE:.0f}",
             fontsize=PLOT_TITLE_FONTSIZE)
fig.tight_layout()
fig.savefig(FIGURES_DIR / f"{NOTEBOOK_NAME}.tissue_quality_map.png", dpi=110)
plt.show()